# Stage 6: Biological interpretation — pathway enrichment
**Group 10 · Computing for Bioinformatics · Person 5 (Kisera Timon Omonde, 166998)**

**Data (Person 1 / 2 / 4 handovers):** GEO GSE15852, Affymetrix U133A microarray, 43 patient-matched tumour/normal pairs; log2-scale expression; paired t-test + BH per gene (exploratory, not count-based RNA-seq DE).

**Research question (from the 4 Nov 2026 minutes):** Can differentially expressed genes in human breast cancer vs. adjacent normal tissue accurately predict tumour status and map onto specific oncogenic signalling pathways?

This notebook answers the *pathway* half of that question. It takes **Person 4's** differential-expression table (`de_results.csv`) and asks which pathways / gene sets are over-represented among the genes that differ between tumour and normal tissue.

| | |
|---|---|
| **Input** | `de_results.csv` (Person 4): gene ID, effect estimate (log2 fold change), p-value, adjusted p-value |
| **Method** | Over-representation analysis (ORA): one-sided hypergeometric test, **explicit background universe**, Benjamini–Hochberg FDR, run separately for up- and down-regulated genes |
| **Gene-set sources** | Reactome, KEGG, GO Biological Process, MSigDB Hallmark (via the Enrichr library mirror in `gseapy`) |
| **Outputs** | `pathway_results.csv`, `pathway_chart_*.png`, `pathway_run_log.json`, optional `interaction_network.png` |

**Scope rules followed:** enrichment is the main analysis; the STRING/NetworkX network is an optional add-on and does not replace it. Findings below are *associations*, not causal claims. If enrichment is inconclusive, the notebook reports that as is.

## 0. Setup

In [ ]:

import sys, subprocess
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
try:
    import gseapy, gprofiler, networkx, statsmodels
except ImportError:
    _pip("gseapy", "gprofiler-official", "networkx", "statsmodels")

In [ ]:
import os, re, json, datetime, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests
import gseapy as gp

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 80, "display.width", 160)
print("gseapy", gp.__version__, "| pandas", pd.__version__, "| run date", datetime.date.today())

## 1. Configuration (edit this cell only)
**Align these settings with Person 4's declared threshold.** The enrichment gene lists must use the *same* significance rule Person 4 reports in Stage 5, otherwise the two sections will contradict each other.

In [ ]:
# ---------- input ----------
DE_PATH = "de_results.csv"          # local copy (used first if present)
DE_URL  = "https://raw.githubusercontent.com/NyamburaWanjohi/Bioinformatics_project/main/data/person4/de_results.csv"
                                    # Person 4's committed table (used if no local copy). Falls back to an upload prompt.

# ---------- gene-list rule: MUST match Person 4's declared threshold ----------
FDR_ALPHA  = 0.05                   # adjusted p-value cut-off for 'differentially expressed'
LFC_CUTOFF = 1.0                    # |log2 fold change| cut-off (set to 0.0 to use FDR only)
# Direction convention: positive log2FC must mean HIGHER IN TUMOUR. Confirm with Person 4.
POSITIVE_LFC_MEANS = "higher in tumour"

# ---------- enrichment settings ----------
ORGANISM_ENRICHR = "Human"
LIBRARIES = {                       # short label -> Enrichr library name
    "Reactome": "Reactome_2022",
    "KEGG":     "KEGG_2021_Human",
    "GO_BP":    "GO_Biological_Process_2023",
    "Hallmark": "MSigDB_Hallmark_2020",
}
MIN_SET_SIZE   = 10                 # gene-set size AFTER restricting to the background
MAX_SET_SIZE   = 500
MIN_OVERLAP    = 3                  # a pathway needs at least this many hits to be reported
ENRICH_ALPHA   = 0.05               # BH-adjusted p-value for calling a pathway 'significant'

# ---------- optional STRING / NetworkX add-on ----------
RUN_NETWORK       = True
NETWORK_N_GENES   = 40              # top genes by |log2FC| among significant genes
STRING_MIN_SCORE  = 700             # high confidence

OUT_DIR = "."                       # outputs are written here

## 2. Load Person 4's results and check them
The checks below are deliberately loud. A silent column mix-up (e.g. reading a raw p-value as an adjusted one) would invalidate everything downstream.

In [ ]:
if os.path.exists(DE_PATH):
    SRC = DE_PATH
else:
    try:
        pd.read_csv(DE_URL, nrows=2)
        SRC = DE_URL
    except Exception:
        try:
            from google.colab import files
            print("Could not read Person 4's file from GitHub. Upload de_results.csv")
            up = files.upload()
            SRC = DE_PATH = list(up.keys())[0]
        except ImportError:
            raise FileNotFoundError("de_results.csv not found locally or at DE_URL.")

de_raw = pd.read_csv(SRC)
print("Loaded", SRC, "->", de_raw.shape)
display(de_raw.head())
print("\nColumns:", list(de_raw.columns))

In [ ]:
def pick(df, candidates, what, required=True):
    low = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    if required:
        raise KeyError(f"No column for {what}. Looked for {candidates}. Columns present: {list(df.columns)}")
    return None

COL_GENE = pick(de_raw, ["gene_id", "gene", "gene_symbol", "symbol", "gene_name", "ensembl_id", "id", "Unnamed: 0"], "gene ID")
COL_LFC  = pick(de_raw, ["log2fc", "log2foldchange", "logfc", "log2_fold_change", "effect", "effect_estimate", "lfc"], "effect estimate (log2 FC)")
COL_P    = pick(de_raw, ["pvalue", "p_value", "pval", "p.value", "p"], "raw p-value")
COL_FDR  = pick(de_raw, ["padj", "fdr", "adj_p", "adj_pval", "adjusted_p", "adj_p_value", "qvalue", "q_value", "adj.p.val"], "adjusted p-value (FDR)")
COL_SYM  = pick(de_raw, ["gene_symbol", "symbol", "gene_name", "hgnc_symbol"], "gene symbol", required=False)
print(f"gene={COL_GENE} | lfc={COL_LFC} | p={COL_P} | fdr={COL_FDR} | symbol={COL_SYM}")

de = de_raw.rename(columns={COL_GENE: "gene_id", COL_LFC: "lfc", COL_P: "p", COL_FDR: "fdr"}).copy()
if COL_SYM and COL_SYM != COL_GENE:
    de = de.rename(columns={COL_SYM: "symbol"})
for c in ["lfc", "p", "fdr"]:
    de[c] = pd.to_numeric(de[c], errors="coerce")

n0 = len(de)
n_nan = int(de[["lfc", "fdr"]].isna().any(axis=1).sum())
de = de.dropna(subset=["lfc", "fdr"])
n_dup = int(de["gene_id"].duplicated().sum())
de = de.sort_values("fdr").drop_duplicates("gene_id", keep="first")
assert (de["fdr"].between(0, 1)).all(), "FDR column has values outside [0, 1]: is it really an adjusted p-value?"
# FDR should never be smaller than the raw p for the same gene (BH only inflates). Catch swapped columns.
if de["p"].notna().any():
    swapped = (de["fdr"] + 1e-12 < de["p"]).mean()
    assert swapped < 0.01, f"{swapped:.1%} of genes have FDR < raw p: columns look swapped."
print(f"Rows in file: {n0} | dropped for missing lfc/FDR: {n_nan} | duplicate gene IDs removed (kept best FDR): {n_dup} | kept: {len(de)}")

## 3. Gene identifiers → HGNC symbols
Enrichment libraries use gene **symbols**. If Person 4's table already has a symbol column it is used as is; if the IDs are Ensembl (`ENSG…`), they are converted with g:Convert and the number lost is reported.

In [ ]:
def looks_ensembl(s):
    return s.astype(str).str.match(r"^ENSG\d+").mean() > 0.8

if "symbol" in de.columns:
    de["symbol"] = de["symbol"].astype(str)
    id_source = "symbol column supplied in de_results.csv"
elif looks_ensembl(de["gene_id"]):
    from gprofiler import GProfiler
    de["ensg"] = de["gene_id"].astype(str).str.replace(r"\.\d+$", "", regex=True)   # strip version suffix
    conv = GProfiler(return_dataframe=True).convert(organism="hsapiens", query=de["ensg"].unique().tolist())
    mapping = (conv[conv["name"].astype(str).str.lower() != "none"]
               .drop_duplicates("incoming").set_index("incoming")["name"])
    de["symbol"] = de["ensg"].map(mapping)
    id_source = "Ensembl IDs converted with g:Convert"
else:
    de["symbol"] = de["gene_id"].astype(str)
    id_source = "gene_id column treated as HGNC symbols"

n_before = len(de)
de = de[de["symbol"].notna() & ~de["symbol"].isin(["nan", "None", ""])].copy()
# several IDs can map to one symbol: keep the most significant
n_multi = int(de["symbol"].duplicated().sum())
de = de.sort_values("fdr").drop_duplicates("symbol", keep="first")
print(f"ID handling: {id_source}")
print(f"Genes with a usable symbol: {len(de)} of {n_before} ({n_before-len(de)} unmapped) | symbol collisions collapsed: {n_multi}")
display(de[["gene_id", "symbol", "lfc", "p", "fdr"]].head())

## 4. Define the gene lists and the background universe
- **Significant genes:** FDR < `FDR_ALPHA` and |log2FC| ≥ `LFC_CUTOFF`, split into *up in tumour* and *down in tumour*.
- **Background universe:** every gene that was actually tested in the DE analysis (after Person 2's filtering). Using the whole genome instead would inflate enrichment because unexpressed genes could never have been detected as significant.

In [ ]:
BACKGROUND = sorted(de["symbol"].unique())
sig  = de[(de["fdr"] < FDR_ALPHA) & (de["lfc"].abs() >= LFC_CUTOFF)]
UP   = sorted(sig.loc[sig["lfc"] > 0, "symbol"])
DOWN = sorted(sig.loc[sig["lfc"] < 0, "symbol"])

gene_lists = pd.DataFrame({
    "set": ["background (all tested)", "significant total", f"up ({POSITIVE_LFC_MEANS})", f"down (lower in tumour)"],
    "n_genes": [len(BACKGROUND), len(sig), len(UP), len(DOWN)],
})
display(gene_lists)
print(f"Rule used: FDR < {FDR_ALPHA} and |log2FC| >= {LFC_CUTOFF}")
if len(UP) + len(DOWN) == 0:
    print("WARNING: no significant genes under this rule. Enrichment cannot be run; report this honestly in Stage 6.")
for name, g in [("UP", UP), ("DOWN", DOWN)]:
    if 0 < len(g) < 15:
        print(f"WARNING: {name} list has only {len(g)} genes: enrichment will have very little power.")
    if len(g) > 0.4 * len(BACKGROUND):
        print(f"WARNING: {name} list is {len(g)/len(BACKGROUND):.0%} of the background: results will be hard to interpret. Consider a stricter cut-off and say so.")

## 5. Enrichment function (over-representation analysis)
For each gene set with *K* genes inside a universe of *N* genes, and a hit list of *n* genes of which *k* fall in the set, the p-value is the upper tail of the hypergeometric distribution, P(X ≥ k). Gene sets are first **restricted to the background**, and the universe *N* is the set of background genes that have at least one annotation in that library. P-values are corrected with **Benjamini–Hochberg across all gene sets tested in that library for that list**.

In [ ]:
def ora(hits, background, gene_sets, min_size=MIN_SET_SIZE, max_size=MAX_SET_SIZE, min_overlap=MIN_OVERLAP):
    # One-sided hypergeometric over-representation test with BH correction.
    bg = set(background)
    hits = set(hits) & bg
    restricted = {t: set(g) & bg for t, g in gene_sets.items()}
    universe = set().union(*restricted.values()) if restricted else set()
    hits = hits & universe                       # hits with no annotation cannot be tested
    N, n = len(universe), len(hits)
    rows = []
    for term, members in restricted.items():
        K = len(members)
        if not (min_size <= K <= max_size):
            continue
        overlap = hits & members
        k = len(overlap)
        p = hypergeom.sf(k - 1, N, K, n) if n > 0 else 1.0
        fold = (k / n) / (K / N) if n > 0 and K > 0 else np.nan
        rows.append((term, K, k, n, N, fold, p, ";".join(sorted(overlap))))
    out = pd.DataFrame(rows, columns=["pathway", "set_size", "overlap", "list_size", "background_size",
                                      "fold_enrichment", "p_value", "overlap_genes"])
    if len(out):
        out["adj_p_value"] = multipletests(out["p_value"].values, method="fdr_bh")[1]
        out["n_sets_tested"] = len(out)
        out = out[out["overlap"] >= min_overlap].sort_values(["adj_p_value", "p_value"]).reset_index(drop=True)
    return out

# ---- unit self-test on a toy example (this is a code check, NOT a result) ----
_bg = [f"G{i}" for i in range(100)]
_sets = {"A": {f"G{i}" for i in range(0, 20)}, "B": {f"G{i}" for i in range(50, 70)}, "C": {f"G{i}" for i in range(80, 95)}}
_hits = [f"G{i}" for i in range(0, 12)]
_t = ora(_hits, _bg, _sets, min_size=5, max_size=50, min_overlap=1).set_index("pathway")
_N = len(set().union(*_sets.values()))          # universe = genes with any annotation (55 here)
_exp = hypergeom.sf(12 - 1, _N, 20, 12)
assert abs(_t.loc["A", "p_value"] - _exp) < 1e-12, "hypergeometric p-value does not match scipy"
assert _t.loc["A", "adj_p_value"] >= _t.loc["A", "p_value"], "BH adjustment must not lower p-values"
print("ORA self-test passed (toy data only).")

## 6. Load gene-set libraries and record exactly what was used

In [ ]:
available = set(gp.get_library_name(organism=ORGANISM_ENRICHR))
missing = {k: v for k, v in LIBRARIES.items() if v not in available}
if missing:
    print("Library names not found in Enrichr (edit LIBRARIES in section 1):", missing)
    # try the newest library that shares the same stem
    for label, name in list(missing.items()):
        stem = re.sub(r"_\d{4}.*$", "", name)
        cand = sorted(a for a in available if a.startswith(stem))
        if cand:
            LIBRARIES[label] = cand[-1]
            print(f"  -> using {cand[-1]} for {label}")
        else:
            LIBRARIES.pop(label)
GENE_SETS = {label: gp.get_library(name, organism=ORGANISM_ENRICHR) for label, name in LIBRARIES.items()}
bg_set = set(BACKGROUND)
for label, gs in GENE_SETS.items():
    annotated = bg_set & set().union(*map(set, gs.values()))
    print(f"{label:9s} {LIBRARIES[label]:32s} {len(gs):5d} gene sets | background genes annotated: {len(annotated)}/{len(bg_set)} ({len(annotated)/len(bg_set):.0%})")
print("Genes with outdated/legacy symbols (e.g. GPR56) will not match current libraries; this lowers coverage slightly and is a limitation.")

## 7. Run the enrichment

In [ ]:
results = []
for direction, hits in [("up_in_tumour", UP), ("down_in_tumour", DOWN)]:
    if len(hits) == 0:
        continue
    for label, gs in GENE_SETS.items():
        r = ora(hits, BACKGROUND, gs)
        if len(r):
            r.insert(0, "library", label)
            r.insert(0, "direction", direction)
            results.append(r)

pathway_results = (pd.concat(results, ignore_index=True) if results else
                   pd.DataFrame(columns=["direction", "library", "pathway", "set_size", "overlap", "list_size",
                                         "background_size", "fold_enrichment", "p_value", "overlap_genes",
                                         "adj_p_value", "n_sets_tested"]))
pathway_results["significant"] = pathway_results["adj_p_value"] < ENRICH_ALPHA
pathway_results.to_csv(os.path.join(OUT_DIR, "pathway_results.csv"), index=False)

print(f"Pathways reported (overlap >= {MIN_OVERLAP}): {len(pathway_results)}")
summary = (pathway_results.groupby(["direction", "library"])["significant"].agg(["size", "sum"])
           .rename(columns={"size": "tested_with_overlap", "sum": f"significant_BH<{ENRICH_ALPHA}"}))
display(summary)

In [ ]:
cols = ["direction", "library", "pathway", "overlap", "set_size", "fold_enrichment", "adj_p_value"]
for direction in ["up_in_tumour", "down_in_tumour"]:
    sub = pathway_results[(pathway_results["direction"] == direction) & pathway_results["significant"]]
    print(f"\n=== {direction}: top significant pathways ===")
    if len(sub):
        display(sub.sort_values("adj_p_value")[cols].head(15))
    else:
        print("No pathway passed the BH threshold for this list.")

## 8. Pathway charts
One chart per library of interest (Hallmark answers the *oncogenic signalling* part of the question; Reactome gives finer detail). If nothing is significant the figure says so instead of showing near-misses as if they were findings.

In [ ]:
def plot_top(library, top_n=10, fname=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, max(3.5, 0.45 * top_n + 1.5)), sharex=False)
    colors = {"up_in_tumour": "#C0392B", "down_in_tumour": "#2471A3"}
    titles = {"up_in_tumour": "Up in tumour", "down_in_tumour": "Down in tumour"}
    any_hit = False
    for ax, direction in zip(axes, ["up_in_tumour", "down_in_tumour"]):
        d = pathway_results[(pathway_results["library"] == library) & (pathway_results["direction"] == direction)
                            & pathway_results["significant"]].sort_values("adj_p_value").head(top_n)
        if d.empty:
            ax.text(0.5, 0.5, f"No {library} pathway significant\n(BH adj. p < {ENRICH_ALPHA})",
                    ha="center", va="center", transform=ax.transAxes, fontsize=11)
            ax.set_axis_off(); ax.set_title(titles[direction]); continue
        any_hit = True
        d = d.iloc[::-1]
        labels = [re.sub(r"\s*\(GO:\d+\)", "", t)[:60] + f"  (k={k})" for t, k in zip(d["pathway"], d["overlap"])]
        ax.barh(labels, -np.log10(d["adj_p_value"].clip(lower=1e-300)), color=colors[direction])
        ax.axvline(-np.log10(ENRICH_ALPHA), ls="--", c="grey", lw=1)
        ax.set_xlabel("-log10 (BH adjusted p)"); ax.set_title(titles[direction])
    fig.suptitle(f"{library} over-representation: tumour vs adjacent normal breast "
                 f"(FDR<{FDR_ALPHA}, |log2FC|>={LFC_CUTOFF})", y=1.02)
    fig.text(0.01, -0.03, "Bars: BH-adjusted hypergeometric p-values. k = genes in both the hit list and the pathway. "
             "Background = all genes tested in DE.", fontsize=8, ha="left")
    fig.tight_layout()
    fig.savefig(fname or os.path.join(OUT_DIR, f"pathway_chart_{library}.png"), dpi=200, bbox_inches="tight")
    plt.show()
    return any_hit

for lib in [l for l in ["Hallmark", "Reactome"] if l in GENE_SETS]:
    plot_top(lib)

## 9. How fragile is the result? (threshold sensitivity)
ORA depends on where the cut-off for "differentially expressed" is drawn. This check re-runs the Hallmark test (or the first available library) under a few rules so the interpretation can state which pathways are robust and which only appear under one cut-off.

In [ ]:
SENS_LIB = "Hallmark" if "Hallmark" in GENE_SETS else next(iter(GENE_SETS))   # falls back if Hallmark is unavailable
print("Sensitivity library:", SENS_LIB)

def hallmark_hits(fdr_a, lfc_c):
    s = de[(de["fdr"] < fdr_a) & (de["lfc"].abs() >= lfc_c)]
    return sorted(s.loc[s["lfc"] > 0, "symbol"]), sorted(s.loc[s["lfc"] < 0, "symbol"])

sens_rows = []
if SENS_LIB:
    for fa, lc in [(FDR_ALPHA, LFC_CUTOFF), (FDR_ALPHA, 0.0), (FDR_ALPHA, 1.5), (0.01, 1.0)]:
        u, d_ = hallmark_hits(fa, lc)
        for dirn, hits in [("up_in_tumour", u), ("down_in_tumour", d_)]:
            r = ora(hits, BACKGROUND, GENE_SETS[SENS_LIB]) if len(hits) else pd.DataFrame()
            sig_terms = r.loc[r["adj_p_value"] < ENRICH_ALPHA, "pathway"].tolist() if len(r) else []
            sens_rows.append({"FDR<": fa, "|log2FC|>=": lc, "direction": dirn, "n_genes": len(hits),
                              "n_sig_sets": len(sig_terms), "top3": "; ".join(sig_terms[:3])})
sens = pd.DataFrame(sens_rows)
display(sens)
sens.to_csv(os.path.join(OUT_DIR, "pathway_sensitivity.csv"), index=False)

## 10. Optional add-on: STRING interaction network
A small, documented network for the strongest differentially expressed genes. This **supports** the enrichment interpretation; it does not replace it. Edges are STRING functional associations (physical or functional), not proof of regulation.

In [ ]:
network_info = {"ran": False}
if RUN_NETWORK and len(sig) > 0:
    try:
        import requests, networkx as nx
        net_genes = sig.reindex(sig["lfc"].abs().sort_values(ascending=False).index).head(NETWORK_N_GENES)
        ids = net_genes["symbol"].tolist()
        base = "https://string-db.org/api/json"
        string_version = requests.get(f"{base}/version", timeout=30).json()[0]["string_version"]
        resp = requests.post(f"{base}/network", timeout=60, data={
            "identifiers": "\r".join(ids), "species": 9606, "required_score": STRING_MIN_SCORE,
            "caller_identity": "group10_bioinformatics_project"})
        resp.raise_for_status()
        edges = pd.DataFrame(resp.json())
        G = nx.Graph()
        G.add_nodes_from(ids)
        if len(edges):
            for a, b, s in edges[["preferredName_A", "preferredName_B", "score"]].itertuples(index=False):
                G.add_edge(a, b, weight=float(s))
        lfc_map = net_genes.set_index("symbol")["lfc"]
        connected = [n for n in G if G.degree(n) > 0]
        H = G.subgraph(connected).copy()
        print(f"STRING v{string_version}: {len(ids)} query genes, {H.number_of_nodes()} connected, {H.number_of_edges()} edges (score >= {STRING_MIN_SCORE})")
        if H.number_of_edges() > 0:
            fig, ax = plt.subplots(figsize=(9, 8))
            pos = nx.spring_layout(H, seed=SEED, k=0.9)
            nx.draw_networkx_edges(H, pos, alpha=0.35, ax=ax)
            nx.draw_networkx_nodes(H, pos, node_size=[120 + 60 * H.degree(n) for n in H],
                                   node_color=[lfc_map.get(n, 0) for n in H], cmap="coolwarm",
                                   vmin=-lfc_map.abs().max(), vmax=lfc_map.abs().max(), ax=ax)
            nx.draw_networkx_labels(H, pos, font_size=7, ax=ax)
            ax.set_title(f"STRING v{string_version} network of top {NETWORK_N_GENES} DE genes (score>={STRING_MIN_SCORE})\n"
                         "node colour = log2FC (red = higher in tumour)")
            ax.set_axis_off()
            fig.savefig(os.path.join(OUT_DIR, "interaction_network.png"), dpi=200, bbox_inches="tight")
            plt.show()
            hubs = sorted(H.degree, key=lambda x: -x[1])[:5]
            print("Highest-degree genes:", hubs)
        network_info = {"ran": True, "string_version": string_version, "n_query": len(ids),
                        "n_connected": H.number_of_nodes(), "n_edges": H.number_of_edges(), "min_score": STRING_MIN_SCORE}
    except Exception as e:
        print("Network step skipped (optional):", type(e).__name__, str(e)[:200])
        network_info = {"ran": False, "error": str(e)[:200]}
else:
    print("Network add-on switched off or no significant genes.")

## 11. Provenance log
Everything another person needs to reproduce or audit this section.

In [ ]:
run_log = {
    "run_date": str(datetime.date.today()),
    "input_file": str(SRC), "n_genes_in_file": int(n0), "n_genes_tested_used_as_background": len(BACKGROUND),
    "id_handling": id_source,
    "gene_list_rule": {"fdr_alpha": FDR_ALPHA, "abs_log2fc_min": LFC_CUTOFF, "positive_lfc_means": POSITIVE_LFC_MEANS},
    "n_up": len(UP), "n_down": len(DOWN),
    "test": "one-sided hypergeometric (ORA), BH FDR within each list x library",
    "set_size_filter": [MIN_SET_SIZE, MAX_SET_SIZE], "min_overlap": MIN_OVERLAP, "enrichment_alpha": ENRICH_ALPHA,
    "libraries": LIBRARIES, "gseapy_version": gp.__version__, "seed": SEED,
    "network": network_info,
}
with open(os.path.join(OUT_DIR, "pathway_run_log.json"), "w") as f:
    json.dump(run_log, f, indent=2)
print(json.dumps(run_log, indent=2))

## 12. Biological interpretation (write this *after* the tables and charts above have run)

> **Instructions for the author.** Replace each `…` with what *your* run produced. Delete any prompt that does not apply. Do not describe a pathway as enriched unless it is in the significant table above, and quote its adjusted p-value and overlap. Never copy text from expectations below as if it were a finding.

**1. What the enrichment shows.**
Under the rule FDR < … and |log2FC| ≥ … , the list contained … up-regulated and … down-regulated genes against a background of … tested genes. Among up-regulated genes, … pathways were significant (BH < 0.05); the strongest were … (adj. p = …, k = …/…). Among down-regulated genes, … pathways were significant; the strongest were … *(If none: "No pathway passed BH correction for this list; the enrichment analysis is inconclusive for this direction.")*

**2. Link back to the research question.**
The oncogenic signalling gene sets in Hallmark that were significant were … (e.g. cell-cycle, proliferation, PI3K/AKT/mTOR, p53, estrogen-response sets, if present). This supports / does not support the idea that the tumour-vs-normal gene differences map onto … signalling.

**3. Compare with published biology.**
Breast tumours are widely reported to show higher proliferation and cell-cycle programmes (e.g. E2F targets, G2-M checkpoint) and lower adipose-related programmes (adipogenesis, fatty-acid metabolism), partly because normal breast tissue is fat-rich. *These are hypotheses to check against your own tables and to cite properly, not results.* My results agree / disagree on … because …

**4. Robustness.**
From the sensitivity table: … pathways stayed significant under all four cut-offs; … appeared only under … .

**5. Network (if run).**
STRING network of the top … genes had … edges; the most connected genes were … . This is consistent with / adds to the pathway result by … ; it does not show regulation.

**6. What this does *not* show.**
- Enrichment is **association**, not causation: a pathway being over-represented does not show it drives the cancer.
- ORA depends on the significance cut-off and on which genes were tested (background).
- Tumour tissue is a mixture of cell types; differences in cell composition (tumour vs stroma/fat/immune) can produce pathway signals unrelated to cancer-cell signalling.
- If the data are not patient-matched, "adjacent normal" differences may include between-patient variation. *(Check Person 1's `dataset_manifest.md` for pairing status.)*
- If Person 4's comparison was on normalized, not raw counts, the gene list is exploratory and so is this enrichment.
- Enrichr libraries are versioned snapshots (see `pathway_run_log.json`).

**7. One-line takeaway for Stage 8 (Conclusions):** …